# **Word2Vec**
직접 만든 Corpus와 `nn.Embedding`을 이용하여 Skip-gram 방식으로 단어 Vector를 학습해보자.


## **1.Corpus 준비하기**

* 비슷한 문맥에서 사용되는 단어를 반복해서 포함한 작은 Corpus를 직접 만든다.

In [5]:
sentences = [
    # Fruit
    "apple is sweet fruit",
    "banana is sweet fruit",
    "grape is sweet fruit",

    "apple is delicious fruit",
    "banana is delicious fruit",
    "grape is delicious fruit",

    "apple tastes fresh",
    "banana tastes fresh",
    "grape tastes fresh",

    "people eat apple",
    "people eat banana",
    "people eat grape",

    # Animal
    "dog is cute animal",
    "cat is cute animal",
    "rabbit is cute animal",

    "dog is friendly pet",
    "cat is friendly pet",
    "rabbit is friendly pet",

    "dog has soft fur",
    "cat has soft fur",
    "rabbit has soft fur",

    "people like dog",
    "people like cat",
    "people like rabbit",

    # Vehicle
    "car is fast vehicle",
    "bus is fast vehicle",
    "train is fast vehicle",

    "car carries people",
    "bus carries people",
    "train carries people",

    "car is transportation",
    "bus is transportation",
    "train is transportation",

    "people ride car",
    "people ride bus",
    "people ride train"
]

In [6]:
print("전체 문장 수 :", len(sentences))
print(*sentences[:5], sep="\n")

전체 문장 수 : 36
apple is sweet fruit
banana is sweet fruit
grape is sweet fruit
apple is delicious fruit
banana is delicious fruit


## **2. Tokenization**

문장을 띄어쓰기 기준으로 Tokenization한다.


In [7]:
# code here
tokens = [text.split() for text in sentences]
print(tokens[0])

['apple', 'is', 'sweet', 'fruit']


##**3. Vocabulary**

각 Token에 고유한 Token ID를 부여한다.


In [8]:
# code here
vocab = {}

for sentence in tokens:
    for word in sentence:
        if vocab.get(word) == None:
            vocab[word] = len(vocab)

print(len(vocab))
print(vocab)


# 출력을 위해 역변환하는 딕셔너리
id_to_word = {idx : word for word, idx in vocab.items()}
print(id_to_word)

30
{'apple': 0, 'is': 1, 'sweet': 2, 'fruit': 3, 'banana': 4, 'grape': 5, 'delicious': 6, 'tastes': 7, 'fresh': 8, 'people': 9, 'eat': 10, 'dog': 11, 'cute': 12, 'animal': 13, 'cat': 14, 'rabbit': 15, 'friendly': 16, 'pet': 17, 'has': 18, 'soft': 19, 'fur': 20, 'like': 21, 'car': 22, 'fast': 23, 'vehicle': 24, 'bus': 25, 'train': 26, 'carries': 27, 'transportation': 28, 'ride': 29}
{0: 'apple', 1: 'is', 2: 'sweet', 3: 'fruit', 4: 'banana', 5: 'grape', 6: 'delicious', 7: 'tastes', 8: 'fresh', 9: 'people', 10: 'eat', 11: 'dog', 12: 'cute', 13: 'animal', 14: 'cat', 15: 'rabbit', 16: 'friendly', 17: 'pet', 18: 'has', 19: 'soft', 20: 'fur', 21: 'like', 22: 'car', 23: 'fast', 24: 'vehicle', 25: 'bus', 26: 'train', 27: 'carries', 28: 'transportation', 29: 'ride'}


## **4. 학습 데이터 만들기**

Skip-gram은 **중심 단어를 이용하여 주변 단어를 예측**한다.
* `window_size=2`이면 중심 단어의 앞뒤 2개 Token을 주변 단어로 사용한다.
* `(중심 단어 ID, 주변 단어 ID)` 형태의 학습 데이터를 만든다.


In [10]:
# code here
window_size = 2
pairs = []

for words in tokens:
    ids = [vocab[word] for word in words]

    for i in range(len(ids)):       # 조합 생성
        center = ids[i]                 # i번째 문자를 center로 잡음
        start = max(0, i - window_size) # 갈 수 있는 범위를 파악
        end = min(len(ids) - 1, i + window_size)

        for j in range(start, end+1):
            if j != i :             # j가 센터(i)가 아니라면
                pairs.append((center, ids[j]))

print(pairs)
print(len(pairs))        

for center, context in pairs[:10]:
    print(f"{id_to_word[center]} -> {id_to_word[context]}")

[(0, 1), (0, 2), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 1), (3, 2), (4, 1), (4, 2), (1, 4), (1, 2), (1, 3), (2, 4), (2, 1), (2, 3), (3, 1), (3, 2), (5, 1), (5, 2), (1, 5), (1, 2), (1, 3), (2, 5), (2, 1), (2, 3), (3, 1), (3, 2), (0, 1), (0, 6), (1, 0), (1, 6), (1, 3), (6, 0), (6, 1), (6, 3), (3, 1), (3, 6), (4, 1), (4, 6), (1, 4), (1, 6), (1, 3), (6, 4), (6, 1), (6, 3), (3, 1), (3, 6), (5, 1), (5, 6), (1, 5), (1, 6), (1, 3), (6, 5), (6, 1), (6, 3), (3, 1), (3, 6), (0, 7), (0, 8), (7, 0), (7, 8), (8, 0), (8, 7), (4, 7), (4, 8), (7, 4), (7, 8), (8, 4), (8, 7), (5, 7), (5, 8), (7, 5), (7, 8), (8, 5), (8, 7), (9, 10), (9, 0), (10, 9), (10, 0), (0, 9), (0, 10), (9, 10), (9, 4), (10, 9), (10, 4), (4, 9), (4, 10), (9, 10), (9, 5), (10, 9), (10, 5), (5, 9), (5, 10), (11, 1), (11, 12), (1, 11), (1, 12), (1, 13), (12, 11), (12, 1), (12, 13), (13, 1), (13, 12), (14, 1), (14, 12), (1, 14), (1, 12), (1, 13), (12, 14), (12, 1), (12, 13), (13, 1), (13, 12), (15, 1), (15, 12), (1, 15), (1,

## **5. Tensor로 변환하기**

중심 단어 ID를 입력 `X`, 주변 단어 ID를 정답 `y`로 사용한다.


In [14]:
import torch

X = torch.tensor([center for center, _ in pairs])
y = torch.tensor([context for _, context in pairs])

## **6. Skip-gram 모델**

`중심 단어 ID → Embedding → Linear → 주변 단어 예측`

* `nn.Embedding`의 Vector가 주변 단어를 잘 예측하는 방향으로 학습된다.
* 학습이 끝난 뒤 `embedding.weight`가 학습된 단어 Vector이다.


In [15]:
import torch.nn as nn

class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        
        # 1. embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        # 2. linear layer - 목표 : 유사한 단어 찾기
        self.linear = nn.Linear(
            embedding_size,
            vocab_size      # 모든 vocab에 대해서 embedding 
        )


    def forward(self, X):
        z = self.embedding(X)
        return self.linear(z)


## **7. 모델 학습**

* 정답 주변 단어의 Token ID를 맞히는 분류 문제이므로 `CrossEntropyLoss`를 사용한다.
* 데이터가 작기 때문에 전체 데이터를 한 번에 학습한다.
- Optimizer로는 Adam을 사용하고, 학습률은 0.001로 설정한다.
- epoch은 약 2000번정도로 설정하자.


In [20]:
model = SkipGram(
    vocab_size= len(vocab),
    embedding_size= 8
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

epochs = 2000

for epoch in range(epochs):
    model.train()
    y_pred = model(X)
    loss = criterion(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 100 == 0:
        print(f"epoch : {epoch}  | loss : {loss:.4f}")

epoch : 99  | loss : 3.3059
epoch : 199  | loss : 3.0703
epoch : 299  | loss : 2.8790
epoch : 399  | loss : 2.7075
epoch : 499  | loss : 2.5425
epoch : 599  | loss : 2.3836
epoch : 699  | loss : 2.2412
epoch : 799  | loss : 2.1233
epoch : 899  | loss : 2.0324
epoch : 999  | loss : 1.9662
epoch : 1099  | loss : 1.9197
epoch : 1199  | loss : 1.8873
epoch : 1299  | loss : 1.8643
epoch : 1399  | loss : 1.8477
epoch : 1499  | loss : 1.8353
epoch : 1599  | loss : 1.8258
epoch : 1699  | loss : 1.8184
epoch : 1799  | loss : 1.8125
epoch : 1899  | loss : 1.8079
epoch : 1999  | loss : 1.8041


## **8. Embedding Vector 확인하기**

* `embedding.weight`에서 각 단어의 학습된 Vector를 확인할 수 있다.


In [ ]:
word = "banana"
word_id = vocab[word]

print(model.embedding.weight[word_id])


tensor([-2.0207,  1.5502, -0.3728,  1.7891, -1.0524,  0.3452, -0.0622, -2.6184],
       grad_fn=<SelectBackward0>)


## **9. Cosine Similarity**

* 학습된 단어 Vector 사이의 유사도를 Cosine Similarity로 비교한다.
* 값이 1에 가까울수록 두 Vector의 방향이 비슷하다.

```
import torch.nn.functional as F
F.cosine_similarity(vector1, vector2)
``


In [26]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

a = model.embedding.weight[vocab["apple"]].detach()
b = model.embedding.weight[vocab["banana"]].detach()

na = np.array([a])
nb = np.array([b])

cosine_similarity(na, nb)

array([[0.76111424]], dtype=float32)

## **10. 비슷한 단어 찾기**

* 하나의 단어와 Cosine Similarity가 높은 단어를 찾아본다.


In [31]:
# code here
def most_similar(word, topk=3):
    vectors = model.embedding.weight.detach() # -> 모든 가중치들이 여기 들어올거다.
    target = vectors[vocab[word]] # 우리의 center 단어

    # 모든 다른 vector와의 거리를 비교해서 점수를 냈다!
    scores = cosine_similarity( 
        np.array([target.detach()]),
        vectors, 
    )

    # topk index
    indexes = torch.topk(torch.tensor(scores), topk+1).indices # 나랑 똑같은 단어 = 1.00
    result = [(id_to_word[i.item()], scores[0][i.item()]) for i in indexes[0]][1:]
    return result

most_similar('car', 3)

[('bus', np.float32(0.9364252)),
 ('train', np.float32(0.83414733)),
 ('vehicle', np.float32(0.733426))]